# Hetionet → Knowledge Graph (KG) Builder

**Source:** [Hetionet ](https://github.com/hetio/hetionet) —  
**Species:** *Homo sapiens*


## Key design decisions per relation type

| Relation | Head ID | Tail ID | Notes |
|---|---|---|---|
| Anatomy_Gene | UBERON ID | NCBI GeneID → Symbol | All 3 Hetionet anatomy subtypes
| Gene_BiologicalProcess | NCBI GeneID → Symbol | GO ID (secondary → primary) | Secondary GO IDs resolved via `GO_SecID_2_prim_ID_dict` |
| Gene_Gene | NCBI GeneID → Symbol | NCBI GeneID → Symbol |
| Gene_MolecularFunction | NCBI GeneID → Symbol | GO ID | Secondary GO IDs resolved |
| Gene_CellularComponent | NCBI GeneID → Symbol | GO ID | Secondary GO IDs resolved |
| Gene_Pathway | NCBI GeneID → Symbol | Reactome pathway ID | No external ontology mapping for Tail |
| Compound_Gene | DrugBank → PubChem CID | NCBI GeneID → Symbol | |
| Compound_SideEffect | DrugBank → PubChem CID | UMLS CUI → SIDER name → HPO ID | Two-step Tail resolution |
| Compound_Disease | DrugBank → PubChem CID | DOID | |
| Compound_Compound | DrugBank → PubChem CID | DrugBank → PubChem CID | |
| Disease_Gene | DOID | NCBI GeneID → Symbol | |
| Disease_Anatomy | DOID | UBERON ID | |
| Disease_Symptom | DOID | MESH/DOID | Tail resolved via DOID, MeSH combined, MeSH supplementary |
| Disease_Disease | DOID | DOID | |

## Output files (13 total)
```
Hetionet/   Anatomy_Gene_Hetionet.csv
Hetionet/   Gene_BiologicalProcess_Hetionet.csv
Hetionet/   Gene_Gene_Hetionet.csv
Hetionet/   Gene_MolecularFunction_Hetionet.csv
Hetionet/   Gene_CellularComponent_Hetionet.csv
Hetionet/   Gene_Pathway_Hetionet.csv
Hetionet/   Compound_Gene_Hetionet.csv
Hetionet/   Compound_SideEffect_Hetionet.csv
Hetionet/   Compound_Disease_Hetionet.csv
Hetionet/   Compound_Compound_Hetionet.csv
Hetionet/   Disease_Gene_Hetionet.csv
Hetionet/   Disease_Anatomy_Hetionet.csv
Hetionet/   Disease_Symptom_Hetionet.csv
Hetionet/   Disease_Disease_Hetionet.csv
```


---
## 0 · Configuration — edit ONLY these two lines

In [65]:
import os
import numpy as np
import pandas as pd
from glob import glob

# ─────────────────────────────────────────────────────────────────────────────
# USER CONFIGURATION
# BASE_PATH : root folder containing all raw input data
# OUT_PATH  : folder where all processed CSVs will be saved
# ─────────────────────────────────────────────────────────────────────────────
BASE_PATH = "/storage/Arushi/090526_EvoAge/kg_formation/data_collection/"
OUT_PATH  = "/storage/Arushi/090526_EvoAge/kg_formation/processed_data/Hetionet/"

# ── Derived input paths (do not edit below this line) ────────────────────────
# PUBCHEM_PATH    = os.path.join(BASE_PATH, "databases_for_mapping/pubchem/CID-Synonym-filtered")
ENS2NCBI_PATH    = os.path.join(BASE_PATH, "databases_for_mapping/ENSEMBL/ENSEMBLE_ID_2_NCBI_Gene_SYM.csv")
NCBI_HUMAN_PATH  = os.path.join(BASE_PATH, "databases_for_mapping/ncbi/Homo_sapiens.gene_info")
UBERON_PATH      = os.path.join(BASE_PATH, "databases_for_mapping/uberon/Uberon_ID_Name_non_dup.csv")
GO_PATH          = os.path.join(BASE_PATH, "databases_for_mapping/MESH/MESH_GO_ID_NAME.csv")
PUBCHEM_DB_PATH  = os.path.join(BASE_PATH, "databases_for_mapping/pubchem/Pubchem_ID_DrugBank_Chebi.csv")
DO_PATH          = os.path.join(BASE_PATH, "databases_for_mapping/DO/All_DO_ref_files.csv")
SIDER_PATH       = os.path.join(BASE_PATH, "databases_for_mapping/sider/side-effect-terms.tsv")
PHENOTYPE_PATH   = os.path.join(BASE_PATH, "ckg/docker_data/neo4j/exports/nodes/Phenotype.csv")
MESH2DOID_PATH   = os.path.join(BASE_PATH, "databases_for_mapping/DO/HumanDiseaseOntology-main/DOreports/MESHinDO.tsv")
MESH_COMB_PATH   = os.path.join(BASE_PATH, "databases_for_mapping/MESH/MESH_Combined.csv")
MESH_SUPP_PATH   = os.path.join(BASE_PATH, "databases_for_mapping/MESH/Mesh_Supplementary_Info.csv")
HETIONET_PATH    = os.path.join(BASE_PATH, "hetionet/hetionet-main/hetnet/tsv/hetionet-v1.0-edges.sif")

os.makedirs(OUT_PATH, exist_ok=True)

print("Paths configured successfully.")
print(f"  Input  root : {BASE_PATH}")
print(f"  Output dir  : {OUT_PATH}")

Paths configured successfully.
  Input  root : /storage/Arushi/090526_EvoAge/kg_formation/data_collection/
  Output dir  : /storage/Arushi/090526_EvoAge/kg_formation/processed_data/Hetionet/


In [41]:
# pd.read_csv('/storage/Arushi/090526_EvoAge/kg_formation/data_collection/ckg/docker_data/neo4j/exports/nodes/Phenotype.tsv',sep = '\t')

# import pandas as pd
# import ast
# import re

# # File path
# file_path = "/storage/Arushi/090526_EvoAge/kg_formation/data_collection/ckg/docker_data/neo4j/exports/nodes/Phenotype.tsv"

# data = []

# with open(file_path, "r", encoding="utf-8") as f:

#     # skip header
#     next(f)

#     for line in f:
        
#         # extract name
#         name_match = re.search(r'name:\s*"([^"]+)"', line)
        
#         # extract id
#         id_match = re.search(r'id:\s*"([^"]+)"', line)

#         if name_match and id_match:
#             data.append({
#                 "name": name_match.group(1),
#                 "id": id_match.group(1)
#             })

# # dataframe
# df = pd.DataFrame(data)

# print(df.head())

# # Optional: save as CSV
# df.to_csv("/storage/Arushi/090526_EvoAge/kg_formation/data_collection/ckg/docker_data/neo4j/exports/nodes/Phenotype.csv", index=False)
# df

---
## 1 · Load reference dictionaries

In [12]:
# ── 1a. ENSEMBL ↔ NCBI gene symbol crosswalk ─────────────────────────────────
ENS_2NCBI = pd.read_csv(ENS2NCBI_PATH)
ENS_2NCBI = ENS_2NCBI[~ENS_2NCBI['name'].isna()]
# {symbol: ENS_ID} — loaded for completeness; not actively used in this notebook
NCBI_2ENS__dict = dict(zip(ENS_2NCBI['name'], ENS_2NCBI['initial_alias']))
del ENS_2NCBI
print(f"Symbol -> Ensembl: {len(NCBI_2ENS__dict):,}")

Symbol -> Ensembl: 40,940


In [13]:
# ── 1b. NCBI Human gene_info ──────────────────────────────────────────────────
# Hetionet uses NCBI GeneIDs (integers) for gene nodes.
# NCBI_ALL_GENEIDname_dict: GeneID (int) -> canonical Symbol
# NCBI_ALL_GENEname_dict:   GeneID (int) -> description
NCBI_ALL_GENE = pd.read_csv(NCBI_HUMAN_PATH, sep='\t')
NCBI_ALL_GENE['ENSEMBLE_ID'] = NCBI_ALL_GENE['Symbol'].map(NCBI_2ENS__dict)
NCBI_ALL_GENEname_dict   = dict(zip(NCBI_ALL_GENE['GeneID'], NCBI_ALL_GENE['description']))
NCBI_ALL_GENEIDname_dict = dict(zip(NCBI_ALL_GENE['GeneID'], NCBI_ALL_GENE['Symbol']))
print(f"NCBI Human GeneID -> Symbol: {len(NCBI_ALL_GENEIDname_dict):,}")

NCBI Human GeneID -> Symbol: 193,505


In [14]:
# ── 1c. UBERON anatomy ontology: ID -> name ───────────────────────────────────
UBERON = pd.read_csv(UBERON_PATH)
UBERON_dict = dict(zip(UBERON['UBERON ID'], UBERON['Name']))
print(f"UBERON anatomy: {len(UBERON_dict):,}")

UBERON anatomy: 15,959


In [19]:
# ── 1d. GO term primary ID -> name ────────────────────────────────────────────
GO = pd.read_csv(GO_PATH)
GO_dict = dict(zip(GO['id'], GO['name']))
print(f"GO terms (primary): {len(GO_dict):,}")

GO terms (primary): 47,995


In [20]:
# ── 1e. GO secondary ID -> primary ID lookup ──────────────────────────────────
# Hetionet GO IDs may be secondary (retired) IDs that map to current primary IDs.
# This dict allows resolving them before looking up names.
GO_SecID = GO[['id','name','namespace','alt_id']].copy()
GO_SecID = GO_SecID[~GO_SecID['alt_id'].isna()]
GO_SecID['alt_id'] = GO_SecID['alt_id'].str.split(r'\s*\|\s*')
GO_SecID = GO_SecID.explode('alt_id')
GO_SecID_2_prim_ID_dict = dict(zip(GO_SecID['alt_id'], GO_SecID['id']))
print(f"GO secondary -> primary ID: {len(GO_SecID_2_prim_ID_dict):,}")

GO secondary -> primary ID: 3,646


In [21]:
# ── 1f. DrugBank ID -> PubChem CID crosswalk ──────────────────────────────────
# Hetionet compound nodes use DrugBank IDs; we resolve to PubChem CID where possible
pubchem = pd.read_csv(PUBCHEM_DB_PATH)
pubchem_DB = pubchem[~pubchem['DRUGBANK_ID'].isna()]
DB2PC_dict = dict(zip(pubchem_DB['DRUGBANK_ID'], pubchem_DB['ID']))  # {DB_ID: CID}
print(f"DrugBank -> PubChem CID: {len(DB2PC_dict):,}")

/tmp/ipykernel_1535588/2965431499.py:3: DtypeWarning: Columns (1,2) have mixed types. Specify dtype option on import or set low_memory=False.
  pubchem = pd.read_csv(PUBCHEM_DB_PATH)


DrugBank -> PubChem CID: 10,702


In [25]:
# ── 1g. Disease Ontology (DO): DOID -> label and DOID -> xrefs ────────────────
DO_All_File = pd.read_csv(DO_PATH)
DOID_disname_dict  = dict(zip(DO_All_File['id'], DO_All_File['label']))
DOID_disAltID_dict = dict(zip(DO_All_File['id'], DO_All_File['xrefs']))
print(f"DO entries: {len(DOID_disname_dict):,}")

DO entries: 11,804


In [26]:
# ── 1h. SIDER side effects: UMLS CUI -> side effect name ─────────────────────
Sider = pd.read_csv(SIDER_PATH, sep='\t')
Sider_dict = dict(zip(Sider['umls_cui_from_meddra'], Sider['side_effect_name']))
print(f"SIDER side effects: {len(Sider_dict):,}")

SIDER side effects: 5,734


In [38]:
# ── 1i. HPO phenotype: name -> HPO ID ─────────────────────────────────────────
# Used for Compound_SideEffect: SIDER name -> HPO ID (two-step lookup)
phenotype = pd.read_csv(PHENOTYPE_PATH, sep=',')
phenotype_dict = dict(zip(phenotype['name'], phenotype['id']))
print(f"HPO entries (name->ID): {len(phenotype_dict):,}")

HPO entries (name->ID): 15,872


In [42]:
# ── 1j. MeSH -> DOID crosswalk ────────────────────────────────────────────────
Mesh2DOID = pd.read_csv(MESH2DOID_PATH, sep='\t')
Mesh2DOID.rename(columns={"ID": "id", "LABEL": "label"}, inplace=True)
Mesh2DOID['xrefs'] = Mesh2DOID['xrefs'].str.split(', ')
Mesh2DOID = Mesh2DOID.explode('xrefs')
Mesh2DOID_dict = dict(zip(Mesh2DOID['xrefs'], Mesh2DOID['id']))
print(f"MeSH -> DOID: {len(Mesh2DOID_dict):,}")

MeSH -> DOID: 3,687


In [50]:
# MESH

In [52]:
# ── 1l. MeSH supplementary concepts: ID -> name ───────────────────────────────
# but Pubchem_Syn_fil_dict is never loaded in this notebook -> NameError.
# Mesh_supp is used only for name lookup (ID->name), so PubChem mapping is removed.
Mesh_supp = pd.read_csv(MESH_SUPP_PATH)
Mesh_supp_dict = dict(zip(Mesh_supp['ID'], Mesh_supp['Name']))
print(f"MeSH supplementary: {len(Mesh_supp_dict):,}")

MeSH supplementary: 324,150


---
## 2 · Load Hetionet edges and build base triple table

In [53]:
# ── 2a. Metaedge abbreviation -> human-readable label ─────────────────────────
relationship_abbreviations = {
    'AdG':  'Anatomy–downregulates–Gene',
    'AeG':  'Anatomy–expresses–Gene',
    'AuG':  'Anatomy–upregulates–Gene',
    'CbG':  'Compound–binds–Gene',
    'CcSE': 'Compound–causes–SideEffect',
    'CdG':  'Compound–downregulates–Gene',
    'CpD':  'Compound–palliates–Disease',
    'CrC':  'Compound–resembles–Compound',
    'CtD':  'Compound–treats–Disease',
    'CuG':  'Compound–upregulates–Gene',
    'DaG':  'Disease–associates–Gene',
    'DdG':  'Disease–downregulates–Gene',
    'DlA':  'Disease–localizes–Anatomy',
    'DpS':  'Disease–presents–Symptom',
    'DrD':  'Disease–resembles–Disease',
    'DuG':  'Disease–upregulates–Gene',
    'GcG':  'Gene–covaries–Gene',
    'GiG':  'Gene–interacts–Gene',
    'GpBP': 'Gene–participates–BiologicalProcess',
    'GpCC': 'Gene–participates–CellularComponent',
    'GpMF': 'Gene–participates–MolecularFunction',
    'GpPW': 'Gene–participates–Pathway',
    'Gr>G': 'Gene-regulates–Gene',
    'PCiC': 'PharmacologicClass–includes–Compound'
}

In [67]:
# ── 2b. Load edge file and parse Head/Tail types ─────────────────────────────
# Hetionet format: source and target columns have format 'EntityType::EntityID'
# e.g. 'Gene::1956'  'Compound::DB00001'  'Anatomy::UBERON:0000002'
Hetionet = pd.read_csv(HETIONET_PATH, sep='\t', dtype=str)
Hetionet['metaedge'] = Hetionet['metaedge'].map(relationship_abbreviations)

# Rename columns to KG schema
Hetionet.rename(columns={
    'source':   'Head',
    'target':   'Tail',
    'metaedge': 'Relation_type'
}, inplace=True)

# Extract entity types from the '::' prefix BEFORE stripping it
Hetionet['Head_type'] = Hetionet['Head'].str.split('::').str[0]
Hetionet['Tail_type'] = Hetionet['Tail'].str.split('::').str[0]

# Build Relation string from entity types (remove spaces for consistency)
Hetionet['Relation'] = (Hetionet['Head_type'] + '_' + Hetionet['Tail_type']).str.replace(' ', '')

Hetionet['KG_Source']  = 'Hetionet'
Hetionet['Head_ID_IS'] = ''
Hetionet['Tail_ID_IS'] = ''

# Select schema columns
desired_cols = ["Head","Relation","Tail","Head_type","Relation_type","Tail_type","Source",
                "KG_Source","Head_ID","Head_Detail_name","Head_ENS","Tail_name","Tail_DOID_Name",
                "Tail_ENS","Head_ID_IS","Tail_ID_IS","Head_Orignal","Tail_Orignal"]
Hetionet = Hetionet[[c for c in desired_cols if c in Hetionet.columns]]

# Strip the '::' prefix to leave only the entity ID in Head and Tail
Hetionet['Head'] = Hetionet['Head'].str.split('::').str[1]
Hetionet['Tail'] = Hetionet['Tail'].str.split('::').str[1]

print(f"Hetionet total edges: {len(Hetionet):,}")
print("\nRelation distribution:")
print(Hetionet['Relation'].value_counts())

Hetionet total edges: 2,250,197

Relation distribution:
Relation
Anatomy_Gene                   726495
Gene_BiologicalProcess         559504
Gene_Gene                      474526
Compound_SideEffect            138944
Gene_MolecularFunction          97222
Gene_Pathway                    84372
Gene_CellularComponent          73566
Compound_Gene                   51429
Disease_Gene                    27977
Compound_Compound                6486
Disease_Anatomy                  3602
Disease_Symptom                  3357
Compound_Disease                 1145
PharmacologicClass_Compound      1029
Disease_Disease                   543
Name: count, dtype: int64


---
## 3 · Helper functions

In [69]:
def resolve_gene_id(df, col):
    """
    Resolve a NCBI GeneID (integer string) column to canonical Gene Symbol.
    Steps:
      1. Convert col to numeric (coerce non-integers to NaN)
      2. Map GeneID -> Symbol (Head_Name / Tail_Name intermediate column)
      3. Map GeneID -> description (detail name)
      4. Swap: Symbol -> col, original GeneID -> {col}_Orignal
      5. Drop rows where Symbol could not be resolved
    """
    df = df.copy()
    name_col   = f'{col}_Name'
    detail_col = f'{col}_Detail_Name'

    df[col]       = pd.to_numeric(df[col], errors='coerce')
    df[name_col]  = df[col].map(NCBI_ALL_GENEIDname_dict)
    df[detail_col]= df[col].map(NCBI_ALL_GENEname_dict)
    df[[name_col, col]] = df[[col, name_col]]   # swap: Symbol -> col, GeneID -> name_col
    before = len(df)
    df = df[~df[col].isna()]
    print(f"  Gene {col} resolved: {len(df):,} kept / {before - len(df):,} dropped")
    return df


def resolve_compound_head(df):
    """
    Resolve DrugBank compound Head to PubChem CID.
    Swaps PubChem CID into Head, original DrugBank ID into Head_pubchem_mapped.
    """
    df = df.copy()
    df['Head_pubchem_mapped'] = df['Head'].map(DB2PC_dict).fillna(df['Head'])
    df['Head_pubchem_mapped'] = df['Head_pubchem_mapped'].astype(str).str.replace(r'\.0$', '', regex=True)
    df[['Head', 'Head_pubchem_mapped']] = df[['Head_pubchem_mapped', 'Head']]
    df['Head_ID_IS'] = 'Pubchem'
    return df


def resolve_compound_tail(df):
    """Same DrugBank -> PubChem CID resolution for Tail column."""
    df = df.copy()
    df['Tail_pubchem_mapped'] = df['Tail'].map(DB2PC_dict).fillna(df['Tail'])
    df['Tail_pubchem_mapped'] = df['Tail_pubchem_mapped'].astype(str).str.replace(r'\.0$', '', regex=True)
    df[['Tail', 'Tail_pubchem_mapped']] = df[['Tail_pubchem_mapped', 'Tail']]
    df['Tail_ID_IS'] = 'Pubchem'
    return df


def resolve_go_tail(df):
    """
    Resolve GO Tail: secondary ID -> primary ID, then primary ID -> GO term name.
    Drops rows where the GO term name cannot be resolved.
    """
    df = df.copy()
    df['Tail'] = df['Tail'].map(GO_SecID_2_prim_ID_dict).fillna(df['Tail'])
    df['Tail_Detail_name'] = df['Tail'].map(GO_dict)
    before = len(df)
    df = df[~df['Tail_Detail_name'].isna()]
    print(f"  GO Tail resolved: {len(df):,} kept / {before - len(df):,} dropped")
    df['Tail_ID_IS'] = 'Quick_GO'
    return df


def select_cols(df, desired):
    """Select only columns that exist in df from desired list."""
    return df[[c for c in desired if c in df.columns]]


def save(df, filename):
    """Save to OUT_PATH and print confirmation."""
    path = os.path.join(OUT_PATH, filename)
    df.to_csv(path, index=False)
    print(f"Saved {len(df):,} rows -> {path}")


print("Helper functions defined.")

Helper functions defined.


---
## 4 · Process and export each relation type

### 4a · Anatomy_Gene

Covers all 3 anatomy–gene metaedges: AdG (downregulates), AeG (expresses), AuG (upregulates).  
Head: UBERON ID → name. Tail: NCBI GeneID → Symbol.

In [70]:
Anatomy_Gene = Hetionet[Hetionet['Relation'] == 'Anatomy_Gene'].copy()

# Head: UBERON ID -> anatomy name (kept as annotation; UBERON ID stays as canonical Head)
Anatomy_Gene['Head_Name'] = Anatomy_Gene['Head'].map(UBERON_dict)
before = len(Anatomy_Gene)
Anatomy_Gene = Anatomy_Gene[~Anatomy_Gene['Head_Name'].isna()]
print(f"UBERON Head resolved: {len(Anatomy_Gene):,} kept / {before - len(Anatomy_Gene):,} dropped")
Anatomy_Gene['Head_ID_IS'] = 'UBERON'

# Tail: NCBI GeneID -> Symbol
Anatomy_Gene = resolve_gene_id(Anatomy_Gene, 'Tail')
Anatomy_Gene['Tail_ID_IS'] = 'NCBI_ID'

print(f"\nAnatomy_Gene: {len(Anatomy_Gene):,} triples")
print("Relation_type values:", set(Anatomy_Gene['Relation_type']))
save(Anatomy_Gene, 'Anatomy_Gene_Hetionet.csv')

UBERON Head resolved: 726,495 kept / 0 dropped
  Gene Tail resolved: 726,156 kept / 339 dropped

Anatomy_Gene: 726,156 triples
Relation_type values: {'Anatomy–downregulates–Gene', 'Anatomy–expresses–Gene', 'Anatomy–upregulates–Gene'}
Saved 726,156 rows -> /storage/Arushi/090526_EvoAge/kg_formation/processed_data/Hetionet/Anatomy_Gene_Hetionet.csv


### 4b · Gene_BiologicalProcess

Head: NCBI GeneID → Symbol. Tail: GO ID (secondary resolved to primary) → GO term name.

In [71]:
Gene_BiologicalProcess = Hetionet[Hetionet['Relation'] == 'Gene_BiologicalProcess'].copy()

Gene_BiologicalProcess = resolve_gene_id(Gene_BiologicalProcess, 'Head')
Gene_BiologicalProcess['Head_ID_IS'] = 'NCBI_ID'
Gene_BiologicalProcess = resolve_go_tail(Gene_BiologicalProcess)

print(f"\nGene_BiologicalProcess: {len(Gene_BiologicalProcess):,} triples")
save(Gene_BiologicalProcess, 'Gene_BiologicalProcess_Hetionet.csv')

  Gene Head resolved: 559,453 kept / 51 dropped
  GO Tail resolved: 559,453 kept / 0 dropped

Gene_BiologicalProcess: 559,453 triples
Saved 559,453 rows -> /storage/Arushi/090526_EvoAge/kg_formation/processed_data/Hetionet/Gene_BiologicalProcess_Hetionet.csv


### 4c · Gene_Gene

Covers GcG (covaries), GiG (interacts), Gr>G (regulates).  
Both Head and Tail: NCBI GeneID → Symbol.

In [72]:
Gene_Gene = Hetionet[Hetionet['Relation'] == 'Gene_Gene'].copy()

Gene_Gene = resolve_gene_id(Gene_Gene, 'Head')
Gene_Gene['Head_ID_IS'] = 'NCBI_ID'
Gene_Gene = resolve_gene_id(Gene_Gene, 'Tail')
Gene_Gene['Tail_ID_IS'] = 'NCBI_ID'

print(f"\nGene_Gene: {len(Gene_Gene):,} triples")
print("Relation_type values:", set(Gene_Gene['Relation_type']))
save(Gene_Gene, 'Gene_Gene_Hetionet.csv')

  Gene Head resolved: 474,488 kept / 38 dropped
  Gene Tail resolved: 474,415 kept / 73 dropped

Gene_Gene: 474,415 triples
Relation_type values: {'Gene–covaries–Gene', 'Gene–interacts–Gene', 'Gene-regulates–Gene'}
Saved 474,415 rows -> /storage/Arushi/090526_EvoAge/kg_formation/processed_data/Hetionet/Gene_Gene_Hetionet.csv


### 4d · Gene_MolecularFunction

In [73]:
Gene_MolecularFunction = Hetionet[Hetionet['Relation'] == 'Gene_MolecularFunction'].copy()

Gene_MolecularFunction = resolve_gene_id(Gene_MolecularFunction, 'Head')
Gene_MolecularFunction['Head_ID_IS'] = 'NCBI_ID'
Gene_MolecularFunction = resolve_go_tail(Gene_MolecularFunction)

print(f"\nGene_MolecularFunction: {len(Gene_MolecularFunction):,} triples")
save(Gene_MolecularFunction, 'Gene_MolecularFunction_Hetionet.csv')

  Gene Head resolved: 97,208 kept / 14 dropped
  GO Tail resolved: 97,208 kept / 0 dropped

Gene_MolecularFunction: 97,208 triples
Saved 97,208 rows -> /storage/Arushi/090526_EvoAge/kg_formation/processed_data/Hetionet/Gene_MolecularFunction_Hetionet.csv


### 4e · Gene_CellularComponent

In [74]:
Gene_CellularComponent = Hetionet[Hetionet['Relation'] == 'Gene_CellularComponent'].copy()

Gene_CellularComponent = resolve_gene_id(Gene_CellularComponent, 'Head')
Gene_CellularComponent['Head_ID_IS'] = 'NCBI_ID'
Gene_CellularComponent = resolve_go_tail(Gene_CellularComponent)

print(f"\nGene_CellularComponent: {len(Gene_CellularComponent):,} triples")
save(Gene_CellularComponent, 'Gene_CellularComponent_Hetionet.csv')

  Gene Head resolved: 73,564 kept / 2 dropped
  GO Tail resolved: 73,564 kept / 0 dropped

Gene_CellularComponent: 73,564 triples
Saved 73,564 rows -> /storage/Arushi/090526_EvoAge/kg_formation/processed_data/Hetionet/Gene_CellularComponent_Hetionet.csv


### 4f · Gene_Pathway

Head: NCBI GeneID → Symbol. Tail: Reactome pathway ID — no external ontology mapping applied.

In [75]:
Gene_Pathway = Hetionet[Hetionet['Relation'] == 'Gene_Pathway'].copy()

Gene_Pathway = resolve_gene_id(Gene_Pathway, 'Head')
Gene_Pathway['Head_ID_IS'] = 'NCBI_ID'
# Tail holds Reactome pathway IDs (e.g. 'R-HSA-xxxxxxx') — no external mapping
Gene_Pathway['Tail_ID_IS'] = 'Reactome'

print(f"\nGene_Pathway: {len(Gene_Pathway):,} triples")
print("Sample Tail (Pathway) IDs:", list(Gene_Pathway['Tail'].head(3)))
save(Gene_Pathway, 'Gene_Pathway_Hetionet.csv')

  Gene Head resolved: 84,349 kept / 23 dropped

Gene_Pathway: 84,349 triples
Sample Tail (Pathway) IDs: ['PC7_6941', 'PC7_5330', 'PC7_6994']
Saved 84,349 rows -> /storage/Arushi/090526_EvoAge/kg_formation/processed_data/Hetionet/Gene_Pathway_Hetionet.csv


### 4g · Compound_Gene

Head: DrugBank ID → PubChem CID. Tail: NCBI GeneID → Symbol.  
Covers CbG (binds), CdG (downregulates), CuG (upregulates).

In [76]:
Compound_Gene = Hetionet[Hetionet['Relation'] == 'Compound_Gene'].copy()

Compound_Gene = resolve_compound_head(Compound_Gene)
Compound_Gene = resolve_gene_id(Compound_Gene, 'Tail')
Compound_Gene['Tail_ID_IS'] = 'NCBI_ID'

before = len(Compound_Gene)
Compound_Gene = Compound_Gene[~Compound_Gene['Head'].isna()]
print(f"\nCompound_Gene: {len(Compound_Gene):,} triples")
print("Relation_type values:", set(Compound_Gene['Relation_type']))
save(Compound_Gene, 'Compound_Gene_Hetionet.csv')

  Gene Tail resolved: 51,428 kept / 1 dropped

Compound_Gene: 51,428 triples
Relation_type values: {'Compound–downregulates–Gene', 'Compound–binds–Gene', 'Compound–upregulates–Gene'}
Saved 51,428 rows -> /storage/Arushi/090526_EvoAge/kg_formation/processed_data/Hetionet/Compound_Gene_Hetionet.csv


### 4h · Compound_SideEffect

Head: DrugBank → PubChem CID.  
Tail: UMLS CUI → SIDER side effect name → HPO ID (two-step lookup).  
Rows where no HPO ID is found are dropped.

**BUG FIX:** Original used `Tail_Orignal` as an intermediate then overwrote it in the swap, making the `~isna()` filter apply to the swapped (now correct) column. Restructured with clear intermediate column names.

In [77]:
Compound_SideEffect = Hetionet[Hetionet['Relation'] == 'Compound_SideEffect'].copy()

Compound_SideEffect = resolve_compound_head(Compound_SideEffect)

# Step 1: UMLS CUI -> SIDER side effect name
Compound_SideEffect['Tail_SIDER_name'] = Compound_SideEffect['Tail'].map(Sider_dict)

# Step 2: SIDER name -> HPO ID
Compound_SideEffect['Tail_HPO_ID'] = Compound_SideEffect['Tail_SIDER_name'].map(phenotype_dict)

# Drop rows where HPO ID could not be resolved
before = len(Compound_SideEffect)
Compound_SideEffect = Compound_SideEffect[~Compound_SideEffect['Tail_HPO_ID'].isna()]
print(f"HPO Tail resolved: {len(Compound_SideEffect):,} kept / {before - len(Compound_SideEffect):,} dropped")

# Swap: HPO ID -> Tail, original UMLS CUI -> Tail_Orignal (audit trail)
Compound_SideEffect['Tail_Orignal'] = Compound_SideEffect['Tail']
Compound_SideEffect['Tail'] = Compound_SideEffect['Tail_HPO_ID']
Compound_SideEffect['Tail_ID_IS'] = 'HP_ID'

print(f"\nCompound_SideEffect: {len(Compound_SideEffect):,} triples")
save(Compound_SideEffect, 'Compound_SideEffect_Hetionet.csv')

HPO Tail resolved: 50,892 kept / 88,052 dropped

Compound_SideEffect: 50,892 triples
Saved 50,892 rows -> /storage/Arushi/090526_EvoAge/kg_formation/processed_data/Hetionet/Compound_SideEffect_Hetionet.csv


### 4i · Compound_Disease

Head: DrugBank → PubChem CID. Tail: DOID. Covers CpD (palliates) and CtD (treats).

In [78]:
Compound_Disease = Hetionet[Hetionet['Relation'] == 'Compound_Disease'].copy()

Compound_Disease = resolve_compound_head(Compound_Disease)
Compound_Disease['Tail_Name'] = Compound_Disease['Tail'].map(DOID_disname_dict)
before = len(Compound_Disease)
Compound_Disease = Compound_Disease[~Compound_Disease['Tail_Name'].isna()]
print(f"DOID Tail resolved: {len(Compound_Disease):,} kept / {before - len(Compound_Disease):,} dropped")
Compound_Disease = Compound_Disease[~Compound_Disease['Head'].isna()]
Compound_Disease['Tail_ID_IS'] = 'DOID'

print(f"\nCompound_Disease: {len(Compound_Disease):,} triples")
print("Relation_type values:", set(Compound_Disease['Relation_type']))
save(Compound_Disease, 'Compound_Disease_Hetionet.csv')

DOID Tail resolved: 1,145 kept / 0 dropped

Compound_Disease: 1,145 triples
Relation_type values: {'Compound–treats–Disease', 'Compound–palliates–Disease'}
Saved 1,145 rows -> /storage/Arushi/090526_EvoAge/kg_formation/processed_data/Hetionet/Compound_Disease_Hetionet.csv


### 4j · Compound_Compound

Both Head and Tail: DrugBank ID → PubChem CID.

In [79]:
Compound_Compound = Hetionet[Hetionet['Relation'] == 'Compound_Compound'].copy()

Compound_Compound = resolve_compound_head(Compound_Compound)
Compound_Compound = resolve_compound_tail(Compound_Compound)
Compound_Compound = Compound_Compound[~Compound_Compound['Head'].isna()]
Compound_Compound = Compound_Compound[~Compound_Compound['Tail'].isna()]

print(f"Compound_Compound: {len(Compound_Compound):,} triples")
save(Compound_Compound, 'Compound_Compound_Hetionet.csv')

Compound_Compound: 6,486 triples
Saved 6,486 rows -> /storage/Arushi/090526_EvoAge/kg_formation/processed_data/Hetionet/Compound_Compound_Hetionet.csv


### 4k · Disease_Gene

Head: DOID. Tail: NCBI GeneID → Symbol. Covers DaG (associates), DdG (downregulates), DuG (upregulates).

In [80]:
Disease_Gene = Hetionet[Hetionet['Relation'] == 'Disease_Gene'].copy()

Disease_Gene['Head_Name'] = Disease_Gene['Head'].map(DOID_disname_dict)
Disease_Gene['Head_ID_IS'] = 'DOID'
Disease_Gene = resolve_gene_id(Disease_Gene, 'Tail')
Disease_Gene['Tail_ID_IS'] = 'NCBI_ID'
Disease_Gene = Disease_Gene[~Disease_Gene['Head'].isna()]

print(f"Disease_Gene: {len(Disease_Gene):,} triples")
print("Relation_type values:", set(Disease_Gene['Relation_type']))
save(Disease_Gene, 'Disease_Gene_Hetionet.csv')

  Gene Tail resolved: 27,936 kept / 41 dropped
Disease_Gene: 27,936 triples
Relation_type values: {'Disease–downregulates–Gene', 'Disease–associates–Gene', 'Disease–upregulates–Gene'}
Saved 27,936 rows -> /storage/Arushi/090526_EvoAge/kg_formation/processed_data/Hetionet/Disease_Gene_Hetionet.csv


### 4l · Disease_Anatomy

Head: DOID. Tail: UBERON ID → anatomy name.

In [81]:
Disease_Anatomy = Hetionet[Hetionet['Relation'] == 'Disease_Anatomy'].copy()

Disease_Anatomy['Head_Name'] = Disease_Anatomy['Head'].map(DOID_disname_dict)
Disease_Anatomy['Tail_Name'] = Disease_Anatomy['Tail'].map(UBERON_dict)
before = len(Disease_Anatomy)
Disease_Anatomy = Disease_Anatomy[~Disease_Anatomy['Tail_Name'].isna()]
Disease_Anatomy = Disease_Anatomy[~Disease_Anatomy['Head_Name'].isna()]
print(f"Resolved: {len(Disease_Anatomy):,} kept / {before - len(Disease_Anatomy):,} dropped")
Disease_Anatomy['Head_ID_IS'] = 'DOID'
Disease_Anatomy['Tail_ID_IS'] = 'UBERON'

print(f"Disease_Anatomy: {len(Disease_Anatomy):,} triples")
save(Disease_Anatomy, 'Disease_Anatomy_Hetionet.csv')

Resolved: 3,602 kept / 0 dropped
Disease_Anatomy: 3,602 triples
Saved 3,602 rows -> /storage/Arushi/090526_EvoAge/kg_formation/processed_data/Hetionet/Disease_Anatomy_Hetionet.csv


### 4m · Disease_Symptom

Head: DOID → name. Tail: symptom ID resolved via 3-source cascade:  
1. DOID (if starts with 'DOID:') 2. MeSH combined 3. MeSH supplementary.

In [82]:
Disease_Symptom = Hetionet[Hetionet['Relation'] == 'Disease_Symptom'].copy()

Disease_Symptom['Head_Name'] = Disease_Symptom['Head'].map(DOID_disname_dict)
before = len(Disease_Symptom)
Disease_Symptom = Disease_Symptom[~Disease_Symptom['Head_Name'].isna()]
print(f"DOID Head resolved: {len(Disease_Symptom):,} kept / {before - len(Disease_Symptom):,} dropped")

# Tail resolution: DOID -> DO label, else MeSH combined, else MeSH supplementary
Disease_Symptom['Tail_name'] = Disease_Symptom['Tail'].apply(
    lambda x: DOID_disname_dict.get(x) if isinstance(x, str) and x.startswith('DOID')
              else (MESH_dict.get(x) or Mesh_supp_dict.get(x))
)
Disease_Symptom['Head_ID_IS'] = 'DOID'
Disease_Symptom['Tail_ID_IS'] = 'MESH'

print(f"Disease_Symptom: {len(Disease_Symptom):,} triples")
save(Disease_Symptom, 'Disease_Symptom_Hetionet.csv')

DOID Head resolved: 3,357 kept / 0 dropped
Disease_Symptom: 3,357 triples
Saved 3,357 rows -> /storage/Arushi/090526_EvoAge/kg_formation/processed_data/Hetionet/Disease_Symptom_Hetionet.csv


### 4n · Disease_Disease

Both Head and Tail: DOID → disease name.

In [83]:
Disease_Disease = Hetionet[Hetionet['Relation'] == 'Disease_Disease'].copy()

Disease_Disease['Head_Name'] = Disease_Disease['Head'].map(DOID_disname_dict)
Disease_Disease['Tail_Name'] = Disease_Disease['Tail'].map(DOID_disname_dict)
before = len(Disease_Disease)
Disease_Disease = Disease_Disease[~Disease_Disease['Head_Name'].isna()]
Disease_Disease = Disease_Disease[~Disease_Disease['Tail_Name'].isna()]
print(f"DOID resolved: {len(Disease_Disease):,} kept / {before - len(Disease_Disease):,} dropped")
Disease_Disease['Head_ID_IS'] = 'DOID'
Disease_Disease['Tail_ID_IS'] = 'DOID'

print(f"Disease_Disease: {len(Disease_Disease):,} triples")
save(Disease_Disease, 'Disease_Disease_Hetionet.csv')

DOID resolved: 543 kept / 0 dropped
Disease_Disease: 543 triples
Saved 543 rows -> /storage/Arushi/090526_EvoAge/kg_formation/processed_data/Hetionet/Disease_Disease_Hetionet.csv


---
## 5 · Summary — all output files

In [84]:
# BUG FIX: original used glob with a hardcoded server path.
# Updated to use OUT_PATH.
print(f"Output files under: {OUT_PATH}\n")
cols = ["Relation","Head_type","Tail_type","Source","KG_Source","Head_ID_IS","Tail_ID_IS"]
df_list = []

for file_path in sorted(glob(os.path.join(OUT_PATH, '*.csv'))):
    try:
        total_df   = pd.read_csv(file_path)
        total_rows = len(total_df)
        temp_df    = total_df.head(5)
        available  = [c for c in cols if c in temp_df.columns]
        row = {'Filename': os.path.basename(file_path), 'no. of triples': total_rows}
        for c in available:
            row[c] = set(temp_df[c].dropna().tolist())
        df_list.append(pd.DataFrame([row]))
    except Exception as e:
        print(f"Error reading {file_path}: {e}")

combined_df = pd.concat(df_list, ignore_index=True)
print(f"Total files: {len(combined_df)}")
combined_df

Output files under: /storage/Arushi/090526_EvoAge/kg_formation/processed_data/Hetionet/

Total files: 14


,Filename,no. of triples,Relation,Head_type,Tail_type,KG_Source,Head_ID_IS,Tail_ID_IS
0,Anatomy_Gene_Hetionet.csv,726156,{Anatomy_Gene},{Anatomy},{Gene},{Hetionet},{UBERON},{NCBI_ID}
1,Compound_Compound_Hetionet.csv,6486,{Compound_Compound},{Compound},{Compound},{Hetionet},{Pubchem},{Pubchem}
2,Compound_Disease_Hetionet.csv,1145,{Compound_Disease},{Compound},{Disease},{Hetionet},{Pubchem},{DOID}
3,Compound_Gene_Hetionet.csv,51428,{Compound_Gene},{Compound},{Gene},{Hetionet},{Pubchem},{NCBI_ID}
4,Compound_SideEffect_Hetionet.csv,50892,{Compound_SideEffect},{Compound},{Side Effect},{Hetionet},{Pubchem},{HP_ID}
5,Disease_Anatomy_Hetionet.csv,3602,{Disease_Anatomy},{Disease},{Anatomy},{Hetionet},{DOID},{UBERON}
6,Disease_Disease_Hetionet.csv,543,{Disease_Disease},{Disease},{Disease},{Hetionet},{DOID},{DOID}
7,Disease_Gene_Hetionet.csv,27936,{Disease_Gene},{Disease},{Gene},{Hetionet},{DOID},{NCBI_ID}
8,Disease_Symptom_Hetionet.csv,3357,{Disease_Symptom},{Disease},{Symptom},{Hetionet},{DOID},{MESH}
9,Gene_BiologicalProcess_Hetionet.csv,559453,{Gene_BiologicalProcess},{Gene},{Biological Process},{Hetionet},{NCBI_ID},{Quick_GO}
